In [48]:
from redis import Redis
redis_client = Redis()

In [49]:
print(redis_client.json().get('umls:C4308010'))


{'cui': 'C4308010', 'embedding': [-0.0016375919, -0.0030410476, -0.001153907, 0.0007871924, -6.669042e-05, -0.001370157, 0.0012437735, 0.0034426725, -0.0040731505, -0.0016658879, 0.00024324519, -0.0034224114, 0.0010790015, -0.0021342533, -0.0009734264, -0.0019466025, 0.0016886787, -0.00055487646, 0.0033884535, -0.003399038, 0.0015733379, -0.0012560847, -0.00075254834, 0.0018307532, 0.00030286858, 0.0039399057, -0.0049844827, 0.0044974145, 0.0020626304, 0.006092656, -0.0073007667, 0.0028743616, -0.007394261, 0.0068227523, -0.005084892, 0.0024055876, 0.010928238, 0.002956952, 0.00037799607, 0.0071149278, -0.00851834, -5.119283e-05, -0.009064694, 0.008269122, -0.00023610031, -0.0033639767, -0.009370481, -0.006265123, -0.0019801483, -0.009252472]}


In [50]:
import json
concepts = json.load(open('data/concepts.json'))
medmentions_documents = json.load(open('data/medmentions.json'))


In [51]:
medmentions_documents[1]

{'id': '25847295',
 'mentions': [{'doc_id': '25847295',
   'end': 43,
   'linked_class': 'UMLS:C0162638',
   'sem_type': 'T038',
   'start': 34,
   'text': 'apoptosis'},
  {'doc_id': '25847295',
   'end': 65,
   'linked_class': 'UMLS:C0085262',
   'sem_type': 'T017',
   'start': 55,
   'text': 'PC12 cells'},
  {'doc_id': '25847295',
   'end': 144,
   'linked_class': 'UMLS:C0150312',
   'sem_type': 'T033',
   'start': 137,
   'text': 'present'},
  {'doc_id': '25847295',
   'end': 219,
   'linked_class': 'UMLS:C0600688',
   'sem_type': 'T037',
   'start': 206,
   'text': 'toxic effects'},
  {'doc_id': '25847295',
   'end': 268,
   'linked_class': 'UMLS:C0162638',
   'sem_type': 'T038',
   'start': 259,
   'text': 'Apoptosis'},
  {'doc_id': '25847295',
   'end': 306,
   'linked_class': 'UMLS:C0229671',
   'sem_type': 'T031',
   'start': 301,
   'text': 'serum'},
  {'doc_id': '25847295',
   'end': 328,
   'linked_class': 'UMLS:C0009968',
   'sem_type': 'T103',
   'start': 322,
   'text': '

In [52]:
testdoc= medmentions_documents[0]
testmention = testdoc['mentions'][0]
text = testdoc['text']
print(text)
print(testmention, text[testmention['start']:testmention['end']])


DCTN4 as a modifier of chronic Pseudomonas aeruginosa infection in cystic fibrosis
Pseudomonas aeruginosa (Pa) infection in cystic fibrosis (CF) patients is associated with worse long-term pulmonary disease and shorter survival, and chronic Pa infection (CPA) is associated with reduced lung function, faster rate of lung decline, increased rates of exacerbations and shorter survival. By using exome sequencing and extreme phenotype design, it was recently shown that isoforms of dynactin 4 (DCTN4) may influence Pa infection in CF, leading to worse respiratory disease. The purpose of this study was to investigate the role of DCTN4 missense variants on Pa infection incidence, age at first Pa infection and chronic Pa infection incidence in a cohort of adult CF patients from a single centre. Polymerase chain reaction and direct sequencing were used to screen DNA samples for DCTN4 variants. A total of 121 adult CF patients from the Cochin Hospital CF centre have been included, all of them carr

In [53]:
import torch
class MedmentionsDataset(torch.utils.data.Dataset):
    def __init__(self, documents, concepts, redis_client):
        self.documents = documents
        self.concepts = concepts
        self.redis_client = redis_client
        self.total_mentions = 0
        self.flat_mentions = []
        self.groups = []
        self.document_index = {}
        for i, document in enumerate(documents):
            self.document_index[document['id']] = i

        for document in documents:
            self.flat_mentions.extend(document['mentions'])
            self.total_mentions += len(document['mentions'])
            for mention in document['mentions']:
                self.groups.append(mention['doc_id'])

    def __len__(self):
        return self.total_mentions

    def __getitem__(self, idx):
      mention = self.flat_mentions[idx]
      concept_id = mention['linked_class'].split(':')[1]
      mention['concept_id'] = concept_id
      ret_item = {}
      ret_item['mention'] = mention
      ret_item['concept'] = self.concepts[concept_id]
      embedding_json = self.redis_client.json().get(f'umls:{concept_id}')
      if embedding_json is not None:
        ret_item['embedding'] = embedding_json['embedding']
      ret_item['context'] = self.documents[self.document_index[mention['doc_id']]]['text']
      return ret_item


    def train_test_val_split(self, seed=42, train_ratio=0.6, test_ratio=0.3):
        from sklearn.model_selection import train_test_split

        # Get unique document IDs
        unique_doc_ids = list(set(self.groups))

        # Split document IDs into train, validation, and test sets
        train_doc_ids, temp_doc_ids = train_test_split(unique_doc_ids, test_size=0.4, random_state=seed)
        valid_doc_ids, test_doc_ids = train_test_split(temp_doc_ids, test_size=0.75, random_state=seed)

        train_documents = [d for d in self.documents if d['id'] in train_doc_ids]
        valid_documents = [d for d in self.documents if d['id'] in valid_doc_ids]
        test_documents = [d for d in self.documents if d['id'] in test_doc_ids]

        train_dataset = MedmentionsDataset(train_documents, self.concepts, self.redis_client)
        valid_dataset = MedmentionsDataset(valid_documents, self.concepts, self.redis_client)
        test_dataset = MedmentionsDataset(test_documents, self.concepts, self.redis_client)

        return train_dataset, valid_dataset, test_dataset



In [54]:
medmentions = MedmentionsDataset(medmentions_documents, concepts, redis_client)

In [55]:
medmentions_train, medmentions_valid, medmentions_test = medmentions.train_test_val_split(seed=42)

In [56]:
instance_test = medmentions_valid[2]
context = instance_test['context']
begin = instance_test['mention']['start']
end = instance_test['mention']['end']

definitions = [definition['text'] for definition in instance_test['concept']['definitions']]
labels = [label['name'] for label in instance_test['concept']['labels']]

print(context[begin:end])
print(instance_test['mention'])
print(definitions)
print(labels)

present
{'doc_id': '25847295', 'end': 144, 'linked_class': 'UMLS:C0150312', 'sem_type': 'T033', 'start': 137, 'text': 'present', 'concept_id': 'C0150312'}
['Being or existing in a specified place or at the specified time.', 'Being or existing in a specified place or at the specified time. (NCI)']
['Present', 'Present', 'Present', 'Present', 'Present', 'Present', 'Present', 'present', 'PRESENT', 'PRESENT', 'In', 'Presence of', 'Presence of', 'Presence', 'of presence', 'Found', 'Present (qualifier value)', 'presente', 'presencia de', 'presente (calificador)']


In [57]:
instance_test['concept']

{'definitions': [{'source': 'NCI',
   'text': 'Being or existing in a specified place or at the specified time.'},
  {'source': 'NCI',
   'text': 'Being or existing in a specified place or at the specified time. (NCI)'}],
 'labels': [{'code': 'https://uts-ws.nlm.nih.gov/rest/content/2017AA/source/RCD/X80xq',
   'concept': 'https://uts-ws.nlm.nih.gov/rest/content/2017AA/CUI/C0150312',
   'name': 'Present',
   'termType': 'PT'},
  {'code': 'https://uts-ws.nlm.nih.gov/rest/content/2017AA/source/SNMI/G-A203',
   'concept': 'https://uts-ws.nlm.nih.gov/rest/content/2017AA/CUI/C0150312',
   'name': 'Present',
   'termType': 'PT'},
  {'code': 'https://uts-ws.nlm.nih.gov/rest/content/2017AA/source/NCI/TCGA',
   'concept': 'https://uts-ws.nlm.nih.gov/rest/content/2017AA/CUI/C0150312',
   'name': 'Present',
   'termType': 'SY'},
  {'code': 'https://uts-ws.nlm.nih.gov/rest/content/2017AA/source/LNC/LA9633-4',
   'concept': 'https://uts-ws.nlm.nih.gov/rest/content/2017AA/CUI/C0150312',
   'name': '

In [58]:

from adapters import PredictionHead
class SpanDetectionHead(PredictionHead, torch.nn.Module):
    def __init__(
        self,
        model,
        head_name,
        **kwargs,
    ):
     super().__init__(head_name)

    def forward(self, outputs, cls_output=None, attention_mask=None, return_dict=False, **kwargs):
      print(outputs)
      pass

class ConceptDenseRepresentationHead(PredictionHead, torch.nn.Module):
    def __init__(
        self,
        model,
        head_name,
        **kwargs,
    ):
     super().__init__(head_name)

    def forward(self, outputs, cls_output=None, attention_mask=None, return_dict=False, **kwargs):
      pass

In [66]:
stop_words = []
with open('stopwordsen.txt', 'r') as f:
    for line in f:
        stop_words.append(line.strip())

termination_terms = []
with open('termination_termsen.txt', 'r') as f:
    for line in f:
        termination_terms.append(line.strip())

def is_stop_or_termination_token(token):
      return token in stop_words or token in termination_terms

def is_span_termination_token(token_span, text):
    return text[token_span[0] : token_span[1]].strip() in termination_terms

In [60]:
def piece_wise_tokenize_token_list(tokenizer, token_list):
    final_token_list = []

    for token in token_list:
        sub_tokens = tokenizer(token)
        final_token_list.append(sub_tokens[-1])

    return tokenizer.convert_tokens_to_ids(final_token_list)


def extract_spans(tokenizer_output):
    """
    Extracts the spans from the tokenizer output
    
    Input:
        - tokenizer_output: output from the tokenizer
        
    Output:
        - final_spans: list of spans for each text
    """
    spans = tokenizer_output["offset_mapping"]
    final_spans = []
    text_final_spans = []
    for i in range(len(spans)):
        for j in range(spans[i].shape[0]):
            span = spans[i][j]
            span_start = span[0].item()
            span_end = span[1].item()
            if j < spans[i].shape[0] - 1 and spans[i][j+1][1].item() != 0:
                text_final_spans.append(torch.tensor([span_start, span_end]))

        final_spans.append(text_final_spans)
    return final_spans


import spacy
# Download en_core_web_md if it isn't already
# !python -m spacy download en_core_web_md
nlp = spacy.load("en_core_web_md")

In [61]:
from adapters import init, AutoAdapterModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import os

tokenizer = AutoTokenizer.from_pretrained("ml6team/keyphrase-extraction-kbir-inspec")
model = AutoAdapterModel.from_pretrained("ml6team/keyphrase-extraction-kbir-inspec")


model.register_custom_head("span_detection_head", SpanDetectionHead)
model.add_custom_head(head_type="span_detection_head", head_name="span_detection_head_supervised")

Some weights of RobertaAdapterModel were not initialized from the model checkpoint at ml6team/keyphrase-extraction-kbir-inspec and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import torch.nn.functional as F
import numpy as np

class LadderLoss(torch.nn.Module):
    """
    Ladder Loss Function for Mention identification
    The distance metric is cosine distance.

    Args:
        margins (list, optional): List of margin values for each ladder component. Default is [0.2].
        thresholds (list, optional): List of threshold values for each ladder component. Default is [].
        betas (list, optional): List of beta values for each ladder component. Default is [].
            Possible relevance degree metrics: BowSim, EcoSTSMatrix, CoMatrix (IoU mode).
        accessories (list, optional): List of additional loss accessories. Each accessory is a tuple
            containing a weight and a callable function. Default is [].
        cumulative (bool, optional): Whether to use cumulative form of inequality in forward pass.
            Default is False.
        debug (bool, optional): Whether to print debug information. Default is False.
    Attributes:
        rd (object): Relevance degree metric.
        hard_negative (bool): Whether to enable hard negative sampling on the first ladder component.
        margins (list): List of margin values for each ladder component.
        thresholds (list): List of threshold values for each ladder component.
        betas (list): List of beta values for each ladder component.
        debug (bool): Whether to print debug information.
        accessories (list): List of additional loss accessories.
        cumulative (bool): Whether to use cumulative form of inequality in forward pass.
    Methods:
        forward(xs, vs, iids, sids, cumulative=False):
            Forward pass of the LadderLoss function.
        accessoryRegression(xs, vs, iids, sids):
            [Optional] Appending a regression loss.
        accessorybinCls(xs, vs, iids, sids):
            [Optional] Accessory: aid VSE training with classification loss.
        accessorySos(xs, vs, iids, sids):
            [Optional] Accessory: aid VSE training with SOSR loss.

    """

    def __init__(self, margins=None, *, thresholds=None, betas=None, reldeg=None, accessories=None, cumulative=False, debug=False):
        """
        Initialize the LadderLoss function.
        Parameters:
        - margins (list): A list of margins for the loss function. Default is [0.2].
        - thresholds (list): A list of thresholds for the loss function. Default is an empty list.
        - betas (list): A list of betas for the loss function. Default is an empty list.
        - accessories (list): A list of tuples representing additional accessories for the loss function. Each tuple should contain a float value and a callable function. Default is an empty list.
        - cumulative (bool): A flag indicating whether to use cumulative loss. Default is False.
        - debug (bool): A flag indicating whether to enable debug mode. Default is False.
        Raises:
        - ValueError: If the number of margins/thresholds does not match the number of betas or if the number of margins/thresholds is greater than 1 and reldeg is not provided.
        Note:
        - Hard negative sampling is enabled by default on the first ladder component (pairwise ranking loss function), but it is mandatory in the rest of the ladder components.
        - The reldeg metric is required if any threshold is set.
        - The accessory functions must have the same signature as the forward function.
        """
        if margins is None:
            margins = [0.2]
        if thresholds is None:
            thresholds = []
        if betas is None:
            betas = []
        if accessories is None:
            accessories = []
        super(LadderLoss, self).__init__()
        self.hard_negative = True
        self.margins = margins
        if len(self.margins) > 1 and (reldeg is None):
            raise ValueError("Missing RelDeg.")
        self.thresholds = thresholds
        if len(self.margins) - 1 != len(self.thresholds):
            raise ValueError("numbers of margin/threshold don't match.")
        elif len(self.thresholds) > 0 and (reldeg is None):
            raise ValueError("where is the reldeg?")
        self.betas = betas
        if len(self.margins) - 1 != len(self.betas):
            raise ValueError("numbers of margin/beta don't match.")
        if len(thresholds) > 0 and (reldeg is None):
            raise ValueError("RelDeg metric is required if set any threshold")
        self.debug = debug
        # check accessory sanity
        if (
            any(not isinstance(x, tuple) for x in accessories)
            or any(len(x) != 2 for x in accessories)
            or any(not isinstance(x[0], float) for x in accessories)
            or any(not isinstance(x[1], callable) for x in accessories)
        ):
            raise ValueError("wrong value for accessories")
        self.accessories = accessories
        self.cumulative = cumulative

    def forward(self, mentions, second_input):
        """
        Perform the forward pass of the model.
        Args:
            xs (torch.Tensor): Input tensor of shape (batch_size, input_size).
            vs (torch.Tensor): Input tensor of shape (batch_size, input_size).
            first_ids (torch.Tensor): Input tensor of shape (batch_size,).
            other_ids (torch.Tensor): Input tensor of shape (batch_size,).
            cumulative (bool, optional): Whether to use cumulative form of inequality. Defaults to False.
        Returns:
            torch.Tensor: The sum of all the losses.
        Raises:
            None
        Notes:
            - This function calculates the forward pass of the model.
            - It computes the scores between the input tensors first_input and second_input.
            - It applies the cumulative form of inequality if cumulative is set to True.
            - It calculates the losses based on the margins and thresholds.
            - It returns the sum of all the losses.

        s(q, q) - s(q, i) > a_1
        s(q, q) - s(q, j) > a_2 + a_1
        s(q, q) - s(q, k) > a+3 + a_2 + a_1
        ...
        """
        device = mentions.device

        # [ First Ladder ]
        scores = torch.mm(mentions, second_input.t())
        diagonal = scores.diag().view(mentions.size(0), 1)
        diag = diagonal.expand_as(scores)
        diagT = diagonal.t().expand_as(scores)
        cost_x2v = (self.margins[0] + scores - diag).clamp(min=0)
        cost_v2x = (self.margins[0] + scores - diagT).clamp(min=0)
        # clear diagonals
        eye = torch.autograd.Variable(torch.eye(scores.size(0))) > 0.5
        eye = eye.to(device)
        cost_x2v = cost_x2v.masked_fill_(eye, 0)
        cost_v2x = cost_v2x.masked_fill_(eye, 0)

        # keep the maximum violating negative for each query
        if self.hard_negative:
            cost_x2v = cost_x2v.max(1)[0]
            cost_v2x = cost_v2x.max(0)[0]

        losses = [cost_x2v.sum() + cost_v2x.sum()]

        # [ l-Ladder (l > 0) and so on ]: Mandatory hard-negatives
        rdmat = torch.tensor(self.rd(other_ids)).float().to(device)
        if self.debug:
            print("LadderLoss>", "sum(STSmat)=", rdmat.sum())
        for l, thre in enumerate(self.thresholds):
            if not self.cumulative:

                simmask = (rdmat >= thre).float()
                dismask = (rdmat < thre).float()
                gt_sim = scores * simmask + 1.0 * dismask
                gt_dis = scores * dismask
                xvld = self.margins[1 + l] - gt_sim.min(dim=1)[0] + gt_dis.max(dim=1)[0]
                xvld = xvld.clamp(min=0)
                vxld = self.margins[1 + l] - gt_sim.min(dim=0)[0] + gt_dis.max(dim=0)[0]
            else:
                # cumulative
                dismask = (rdmat < thre).float()
                gt_dis = scores * dismask
                xvld = (
                    np.sum(self.margins[: l + 2])
                    - diagonal.view(-1)
                    + gt_dis.max(dim=1)[0]
                )
                xvld = xvld.clamp(min=0)
                vxld = (
                    np.sum(self.margins[: l + 2])
                    - diagonal.view(-1)
                    + gt_dis.max(dim=0)[0]
                )
            vxld = vxld.clamp(min=0)

            losses.append(self.betas[l] * (xvld.sum() + vxld.sum()))
        # deal with the additional loss accessories
        losses.extend(
            weight * call(mentions, second_input, first_ids, other_ids) for weight, call in self.accessories
        )
        return sum(losses)

    def accessoryRegression(self, xs, vs, iids, sids):
        """
        [optional] Appending a regression loss.
        """
        rdmat = torch.tensor(self.rd(sids)).float().to(xs.device)
        xvs = F.cosine_similarity(xs[:, :, None], vs.T[None, :, :])
        return ((xvs - rdmat) ** 2).mean()

    def accessorybinCls(self, xs, vs, iids, sids):
        """
        Accessory: aid VSE training with classification loss
        NOTE: must use the same function signature as self.forward
        """
        raise NotImplementedError

    def accessorySos(self, xs, vs, iids, sids):
        """
        Accessory: aid VSE training with SOSR loss
        reference:
        """
        raise NotImplementedError

In [62]:
# prompt: Please write a pytorch lighting training loop for the adapter model using the medmentions corpus that's already loaded

import pytorch_lightning as pl
from torch.utils.data import DataLoader

class LitAdapterModel(pl.LightningModule):
    def __init__(self, model, learning_rate=1e-4):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate

    def forward(self, input_ids, attention_mask):
        print(input_ids)
        return self.model(input_ids=input_ids, attention_mask=attention_mask)

    def training_step(self, batch, batch_idx):
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        outputs = self(input_ids, attention_mask)
        
        
        # Define your loss function here
        loss = .1
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        outputs = self(input_ids, attention_mask)
        # Define your validation metrics here
        val_loss = .1
        self.log('val_loss', val_loss)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)


def first_non_zero_embedding_size(embeddings):
    return next(
        (
            len(embedding)
            for embedding in embeddings
            if embedding is not None and len(embedding) > 0
        ),
        0,
    )

# Prepare data for training
def collate_fn(batch):
    context_texts = [item['context'] for item in batch]
    mention_spans = [torch.tensor([item['mention']['start'],item['mention']['start']]) for item in batch]
    mention_texts = [item['context'][item['mention']['start']:item['mention']['end']] for item in batch]
    label_texts = [[label['name'] for label in item['concept']['labels']] for item in batch]
    definition_texts = [[label['name'] for label in item['concept']['labels']] for item in batch]

    # Split the context texts into sentences using spacy
    context_docs = [nlp(ctext) for ctext in context_texts]
    context_texts = []
    for doc_i in range(len(context_docs)):
        sent_index = 0
        for sent in context_docs[doc_i].sents:
            if (
                mention_spans[doc_i][0] >= sent.start_char
                and mention_spans[doc_i][1] <= sent.end_char
            ):
                break
            sent_index += 1
        # print("Sent: ", sent_index)
        context_texts.append(list(context_docs[doc_i].sents)[sent_index].text)

    encoded_contexts = tokenizer(context_texts, return_offsets_mapping=True, truncation=True, padding=True, return_tensors="pt")
    spans = extract_spans(encoded_contexts)

    encoded_mentions = tokenizer(mention_texts, truncation=True, padding=True, return_tensors="pt")

    encoded_labels = [tokenizer(labels, truncation=True, padding=True, return_tensors="pt") for labels in label_texts]
    encoded_definitions = [tokenizer(definitions, truncation=True, padding=True, return_tensors="pt") for definitions in definition_texts]
    concept_embeddings = [item['embedding'] if 'embedding' in item else None for item in batch]

    emb_size = first_non_zero_embedding_size(concept_embeddings)
    concept_embeddings = torch.tensor([embedding if embedding is not None else torch.zeros(emb_size) for embedding in concept_embeddings])

    return {
        "context": context_texts,
        "mention": mention_texts,
        "label": label_texts,
        "definition": definition_texts,
        "concept_embeddings": concept_embeddings,
        "mention_spans": mention_spans,
        "context_input_ids": encoded_contexts["input_ids"],
        "context_attention_mask": encoded_contexts["attention_mask"],
        "context_token_spans": spans,
        "mention_input_ids": encoded_mentions["input_ids"],
        "mention_attention_mask": encoded_mentions["attention_mask"],
        "label_input_ids": [label["input_ids"] for label in encoded_labels],
        "label_attention_mask": [label["attention_mask"] for label in encoded_labels],
        "definition_input_ids": [
            definition["input_ids"] for definition in encoded_definitions
        ],
        "definition_attention_mask": [
            definition["attention_mask"] for definition in encoded_definitions
        ],
    }

In [63]:
valid_dataloader = DataLoader(medmentions_valid, batch_size=16, shuffle=False, collate_fn=collate_fn)

it = iter(valid_dataloader)
batch = next(it)

#Pretty print batch using pprint
from pprint import pprint

batch

mention_spans = batch['mention_spans']

context_token_spans = batch['context_token_spans']
context_texts = batch['context']

mentions = batch['mention']

mention_input_ids = batch['mention_input_ids']
mention_attention_mask = batch['mention_attention_mask']

context_input_ids = batch['context_input_ids']
context_attention_mask = batch['context_attention_mask']

mention_embeddings = model(input_ids=mention_input_ids, attention_mask=mention_attention_mask)
context_embeddings = model(input_ids=context_input_ids, attention_mask=context_attention_mask)

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-2.9076e-01, -1.3535e-01, -1.6907e-01,  ..., -4.8074e-02,
           1.0461e-01,  3.9682e-01],
         [ 6.1582e-01, -7.0279e-01, -8.0592e-01,  ...,  1.0050e+00,
           6.9165e-01,  6.1879e-01],
         [-8.8732e-01, -5.2688e-01, -2.7251e-01,  ...,  9.3894e-01,
          -2.8894e-01,  6.8505e-01],
         [-1.2926e-01, -2.1972e-01,  1.8244e-01,  ...,  1.8928e-01,
           1.3013e-01, -1.9411e-01],
         [-1.4624e+00, -5.1123e-02,  6.3922e-01,  ...,  3.9764e-02,
           1.7545e-01, -1.0002e+00],
         [-1.4624e+00, -5.1123e-02,  6.3922e-01,  ...,  3.9764e-02,
           1.7545e-01, -1.0002e+00]],

        [[-5.2447e-01, -2.1823e-01, -3.7975e-01,  ...,  2.0749e-02,
           2.8509e-01,  5.9181e-01],
         [ 2.9793e-02, -3.0106e-01, -5.5166e-01,  ...,  1.3732e+00,
          -2.1049e-01,  9.6364e-01],
         [-6.6079e-01, -7.8540e-02, -4.4067e-01,  ...,  3.0383e-01,
          -4.5443e-01,  1.0

In [64]:
print(len(mention_spans), len(context_token_spans), len(mentions))
print(context_token_spans[0])

16 16 16
[tensor([0, 0]), tensor([0, 1]), tensor([1, 4]), tensor([4, 5]), tensor([5, 9]), tensor([ 9, 11]), tensor([12, 16]), tensor([16, 17]), tensor([17, 19]), tensor([19, 21]), tensor([21, 24]), tensor([25, 33]), tensor([34, 39]), tensor([39, 43]), tensor([44, 51]), tensor([52, 54]), tensor([55, 57]), tensor([57, 59]), tensor([60, 65]), tensor([65, 66]), tensor([66, 67]), tensor([67, 70]), tensor([70, 71]), tensor([71, 75]), tensor([75, 77]), tensor([78, 81]), tensor([82, 87]), tensor([87, 88]), tensor([88, 93]), tensor([94, 95]), tensor([95, 98]), tensor([98, 99]), tensor([ 99, 103]), tensor([103, 105]), tensor([106, 109]), tensor([109, 111]), tensor([111, 113]), tensor([113, 117]), tensor([118, 122]), tensor([123, 125]), tensor([126, 128]), tensor([128, 129]), tensor([130, 131]), tensor([131, 132]), tensor([133, 136]), tensor([137, 144]), tensor([145, 147]), tensor([148, 155]), tensor([156, 167]), tensor([168, 170]), tensor([171, 181]), tensor([182, 194]), tensor([194, 195]), tens

In [65]:
def extract_mention_candidates(context_token_spans, context_embeddings, context_text):
    current_token_span_index = 0
    while current_token_span_index < len(context_token_spans):
        # we get the current token span
        current_span = context_token_spans[current_token_span_index]
        concept_start = current_span[0]
        concept_end = current_span[1]

        current_text = context_text[concept_start:concept_end]
        if not is_stop_or_termination_token(current_text.strip()):
            pass

def extract_subsequence(current_span, context_token_spans, context_embeddings, context_text):
    concept_start = current_span[0]
    # For now we have matched a single terms, so currently the end position will be that of the current token
    concept_end = current_span[1]
    match_cursor = 0
    stop_count = 0
    current_token_span_index = 0
    while current_token_span_index + match_cursor < len(
        context_token_spans
    ) and not is_span_termination_token(
        context_token_spans[current_token_span_index + match_cursor], context_text
    ):
        # We get the next token and position span
        next_span = context_token_spans[current_token_span_index + match_cursor]
        next_token = context_text[next_span[0] : next_span[1]]
        #  if the token is in the termination list the matching process ends here
        if next_token.strip() not in stop_words:
            text_up_to_now = context_text[concept_start : next_span[1]]
        else:
            stop_count += 1

SyntaxError: incomplete input (1843441778.py, line 11)

In [ ]:
candidates = extract_mention_candidates(
    context_token_spans[0], context_embeddings["last_hidden_state"][0], context_texts[0]
)

In [ ]:
train_dataloader = DataLoader(medmentions_train, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(medmentions_valid, batch_size=16, shuffle=False, collate_fn=collate_fn)

# Initialize the Lightning module
lit_model = LitAdapterModel(model)

# Train the model
trainer = pl.Trainer(max_epochs=10)  # Adjust max_epochs as needed
trainer.fit(lit_model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

/home/atchechmedjiev/anaconda3/envs/pyclinrec/lib/python3.11/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/atchechmedjiev/anaconda3/envs/pyclinrec/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default

  | Name  | Type                | Params | Mode
-----------------------------------------------------
0 | model | RobertaAdapterModel | 355 M  | eval


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/atchechmedjiev/anaconda3/envs/pyclinrec/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

KeyError: 'input_ids'